# Debugging Guide

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google/orbax/blob/main/docs/guides/checkpoint/v1/debug_guide.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/google/orbax/blob/main/docs/guides/checkpoint/v1/debug_guide.ipynb) [![View on GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/google/orbax/blob/main/docs/guides/checkpoint/v1/debug_guide.ipynb)

Inspect checkpoint layout, on-disk vs. implied size, PyTree metadata, and individual parameter slices with the Orbax V1 API (`from orbax.checkpoint import v1 as ocp`). For common errors and distributed debugging, see [Common V1 Errors & Troubleshooting](https://orbax.readthedocs.io/en/latest/guides/checkpoint/v1/troubleshooting.html).

> **Legacy V0 checkpoints:** See [How to use V1 API to load checkpoints saved with V0 API](https://orbax.readthedocs.io/en/latest/guides/checkpoint/v1/orbax_v0_to_v1_migration.html#how-to-use-v1-api-to-load-checkpoints-saved-with-v0-api) or the [Legacy Debugging Guide](https://orbax.readthedocs.io/en/latest/guides/checkpoint/debug_guide.html).

## Setup

### Imports

In [ ]:
import asyncio
import collections
import operator
from etils import epath
import jax
import numpy as np
from orbax.checkpoint import v1 as ocp
import tensorstore as ts

### Create Sample Checkpoint

Create a small sample checkpoint for the inspection cells below; enter your own checkpoint path in any `path` parameter cell to debug an existing checkpoint.

In [ ]:
mesh = jax.sharding.Mesh(jax.devices(), ('x',))
sharding = jax.sharding.NamedSharding(mesh, jax.sharding.PartitionSpec('x'))

state = {
    'a': {
        'x': jax.device_put(np.arange(2**22, dtype=np.int32), sharding),
        'y': jax.device_put(np.arange(1024, dtype=np.int32), sharding),
    },
    'b': np.ones(8, dtype=np.float32),
    'c': 42,
}

default_param_name = 'a.x'
default_path = epath.Path('/tmp/v1_debug_guide_checkpoint')
default_path.rmtree(missing_ok=True)
ocp.save(default_path, state)

## Layout & Metadata

Validate a checkpoint with `ocp.metadata(path)` (which raises `ocp.errors.InvalidLayoutError` on non-checkpoints) and inspect its checkpointables, save duration, and directory contents. See the [Checkpoint Format Guide](https://orbax.readthedocs.io/en/latest/guides/checkpoint/v1/checkpoint_format.html) for directory layout details.

In [ ]:
path = ""  # @param {type:"string"}
path = epath.Path(path) if path else default_path

# Raises InvalidLayoutError if `path` is not a valid checkpoint
# (unlike `ocp.is_orbax_checkpoint`, which accepts empty directories).
_ = ocp.metadata(path)
chkpt_meta = ocp.checkpointables_metadata(path)

print('Checkpointables:', list(chkpt_meta.metadata.keys()))
if chkpt_meta.init_timestamp_nsecs and chkpt_meta.commit_timestamp_nsecs:
  secs = (chkpt_meta.commit_timestamp_nsecs - chkpt_meta.init_timestamp_nsecs) / 1e9
  print(f'Save duration:   {secs:.3f} s')
print('Custom metadata:', chkpt_meta.custom_metadata)
print('Contents:', [p.name + ('/' if p.is_dir() else '') for p in sorted(path.iterdir())])

## Checkpoint Size

### Actual Size on Disk

Measure the physical size of all files and subdirectories under `path`.

In [ ]:
async def disk_usage(target_path: epath.Path) -> int:
  """Returns total on-disk bytes for a shallow checkpoint directory."""

  async def helper(p: epath.Path) -> int:
    if p.is_dir():
      return await disk_usage(p)
    try:
      return (await asyncio.to_thread(p.stat)).length
    except FileNotFoundError:
      return 0

  return sum(await asyncio.gather(*(helper(p) for p in target_path.iterdir())))


disk_bytes = await disk_usage(path)
disk_bytes_path = path
print(f'Actual size: {disk_bytes / 1e9:.3f} GB ({disk_bytes:,} bytes)')

### Implied Size from Checkpoint Metadata

Compare on-disk size against the uncompressed byte size implied by `ocp.metadata(path)`. On-disk size is typically smaller due to ZSTD compression; a larger footprint usually indicates granular chunking, extra checkpointables, or leftover temporary files (see [Maximizing Performance](https://orbax.readthedocs.io/en/latest/guides/checkpoint/v1/maximizing_performance.html)).

In [ ]:
custom_path = ""  # @param {type:"string"}
path = epath.Path(custom_path) if custom_path else globals().get('path', default_path)

In [ ]:
metadata = ocp.metadata(path).metadata
dtype_counts = collections.defaultdict(int)
non_array_leaves = 0


def get_arr_bytes(meta) -> int:
  global non_array_leaves
  if (
      hasattr(meta, 'dtype')
      and hasattr(meta, 'shape')
      and meta.dtype is not None
      and meta.shape is not None
  ):
    dtype = np.dtype(meta.dtype)
    dtype_counts[str(dtype)] += 1
    return int(np.prod(meta.shape, dtype=np.int64)) * dtype.itemsize
  non_array_leaves += 1
  return 0


total_bytes = jax.tree.reduce(
    operator.add, jax.tree.map(get_arr_bytes, metadata), initializer=0
)
print(f'Implied size: {total_bytes / 1e9:.3f} GB ({total_bytes:,} bytes)')
if globals().get('disk_bytes_path') == path and disk_bytes > 0 and total_bytes > 0:
  print(f'Compression ratio (implied / on-disk): {total_bytes / disk_bytes:.2f}x')
print('Dtype counts:', dict(sorted(dtype_counts.items())))
if non_array_leaves:
  print('Non-array leaves:', non_array_leaves)

## Tree Metadata

List saved parameter keys, shapes, and dtypes to diagnose missing or renamed leaves, then cross-reference them against TensorStore (string leaves are stored in `_strings.json` rather than as array entries).

In [ ]:
custom_path = ""  # @param {type:"string"}
path = epath.Path(custom_path) if custom_path else globals().get('path', default_path)

In [ ]:
def _key_to_str(key) -> str:
  for attr in ('key', 'idx', 'name'):
    if hasattr(key, attr):
      return str(getattr(key, attr))
  return str(key)


metadata = ocp.metadata(path).metadata
flat_metadata = {
    '.'.join(_key_to_str(k) for k in kp): v
    for kp, v in jax.tree_util.tree_flatten_with_path(metadata)[0]
}
for name, val in flat_metadata.items():
  info = (
      f'shape={val.shape}, dtype={val.dtype}'
      if hasattr(val, 'shape')
      else f'type={type(val).__name__}'
  )
  print(f'  {name}: {info}')

In [ ]:
def resolve_pytree_dir(ckpt_path: epath.Path) -> epath.Path:
  if (ckpt_path / 'state' / '_METADATA').exists():
    return ckpt_path / 'state'
  for child in sorted(ckpt_path.iterdir()):
    if child.is_dir() and (child / '_METADATA').exists():
      return child
  raise FileNotFoundError(f'No PyTree _METADATA found under {ckpt_path}')


def make_kvstore_base_url(target_dir: epath.Path) -> str:
  try:
    return target_dir.as_uri()
  except ValueError:
    return target_dir.resolve().as_uri()


pytree_path = resolve_pytree_dir(path)
if (pytree_path / 'manifest.ocdbt').exists():
  ts_kv = ts.KvStore.open(
      {'driver': 'ocdbt', 'base': make_kvstore_base_url(pytree_path)}
  ).result()
  ts_contents = [
      p.removesuffix('/zarr.json').removesuffix('/.zarray')
      for p in (k.decode('utf-8') for k in ts_kv.list().result())
      if p.endswith(('/zarr.json', '/.zarray'))
  ]
else:
  ts_contents = [
      c.name for c in sorted(pytree_path.iterdir())
      if c.is_dir() and ((c / 'zarr.json').exists() or (c / '.zarray').exists())
  ]

# String leaves are stored in _strings.json rather than as OCDBT/Zarr arrays.
expected = [k for k, v in flat_metadata.items() if not isinstance(v, str)]
assert sorted(expected) == sorted(ts_contents), (sorted(expected), sorted(ts_contents))
print('TensorStore entries match metadata:', sorted(ts_contents))

## Individual Parameters

In [ ]:
custom_path = ""  # @param {type:"string"}
param_name = ""  # @param {type:"string"}  # Obtain from Tree Metadata above.
path = epath.Path(custom_path) if custom_path else globals().get('path', default_path)
param_name = param_name or default_param_name

### Value Metadata

Inspect shape, dtype, saved sharding, and chunk shapes. Read `sharding_metadata` rather than `.sharding` when inspecting from a different device topology.

In [ ]:
flat_metadata = {
    '.'.join(_key_to_str(k) for k in kp): v
    for kp, v in jax.tree_util.tree_flatten_with_path(ocp.metadata(path).metadata)[0]
}
value_meta = flat_metadata[param_name]
print(f'Parameter: {param_name}')
print(f'Shape:     {getattr(value_meta, "shape", "scalar")}')
print(f'Dtype:     {getattr(value_meta, "dtype", type(value_meta).__name__)}')
if getattr(value_meta, 'sharding_metadata', None) is not None:
  print(f'Sharding:  {value_meta.sharding_metadata}')
if getattr(value_meta, 'storage_metadata', None) is not None:
  sm = value_meta.storage_metadata
  print(f'Chunks:    chunk_shape={sm.chunk_shape}, write_shape={sm.write_shape}')

### Array Value

Read a small slice directly via TensorStore's [`auto` driver](https://google.github.io/tensorstore/driver/auto/index.html) without loading the full array into host RAM (slice every axis within bounds to avoid OOM). For partial loading or resharding onto a different topology, see [Model Surgery (`ocp.PLACEHOLDER`)](https://orbax.readthedocs.io/en/latest/guides/checkpoint/v1/model_surgery.html#placeholder) and [Loading with Resharding](https://orbax.readthedocs.io/en/latest/guides/checkpoint/v1/maximizing_performance.html#loading-with-resharding).

In [ ]:
pytree_path = resolve_pytree_dir(path)
kvstore = (
    {'driver': 'ocdbt', 'base': make_kvstore_base_url(pytree_path), 'path': param_name}
    if (pytree_path / 'manifest.ocdbt').exists()
    else make_kvstore_base_url(pytree_path / param_name)
)
t = ts.open({'driver': 'auto', 'kvstore': kvstore}, open=True).result()
slices = tuple(slice(0, min(dim, 10)) for dim in t.shape)
print((t[slices] if slices else t).read().result())